In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

In [2]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

print(fixed.x_cg_max, fixed.x_cg_min, fixed.x_LE_wing, fixed.z_cg, fixed.z_tail_cone)
fixed.x_LE_wing = 1.255
delta_z = 0.01
fixed.z_tail_cone += delta_z
fixed.z_cg += delta_z
print(fixed.z_cg, fixed.z_tail_cone)
fixed.fuel_mass = 13.54
fixed.x_cg_min = 1.336 #m
fixed.x_cg_max = 1.403 #m
fixed.mass = 42.2 #kg 

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

1.3589152301106575 1.271545179962353 1.4400000000000002 0.17957635338517247 0.11000000000000004
0.18957635338517248 0.12000000000000004


# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [3]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [4]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [5]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

fuselage_diameter = fixed.fuselage.diameter_max
size_planform(planform=standard_wing, thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=fuselage_diameter, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

Stresses 42860371.45245816, 605594930.7107136, 0.0004
Stresses 28624990.457300115, 415981846.4171284, 0.0005959183673469389
Stresses 21433921.70716872, 320415537.54634386, 0.0007918367346938775
Stresses 17095508.328990243, 262939334.64787665, 0.0009877551020408164
Stresses 14193259.379450185, 224643733.746665, 0.0011836734693877551
Stresses 12115317.823862324, 197360696.28089306, 0.0013795918367346938


In [6]:
print(standard_wing.mass_cache, standard_wing.x_cg_cache, fixed.fuel_mass)

0.25583403427157053 0.21700554646188894 13.54


# We consider the thing to be tailed

In [7]:
tail = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=5., taper_h=.7).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

Stresses 20441698.76918255, 66847197.21362256, 0.0004
Stresses 13697440.683759186, 46725634.097132504, 0.0005959183673469389
Stresses 9220631.337344704, 32564554.37648753, 0.0004
Stresses 6133590.840228666, 22044681.892620567, 0.0005959183673469389
Stresses 26879022.237714637, 86439190.26077685, 0.0004
Stresses 18010549.893404793, 59780112.89434845, 0.0005959183673469389
Stresses 12272181.608042996, 24681342.545623142, 0.0004
Stresses 8195369.497276261, 16707532.781250859, 0.0005959183673469389
Stresses 6135948.946476572, 12680591.602760851, 0.0007918367346938775
Stresses 26695984.946217332, 85752047.54779415, 0.0004
Stresses 17887904.597506214, 59318173.55624765, 0.0005959183673469389
Stresses 12186443.653323453, 24899458.440671757, 0.0004
Stresses 8137529.719751075, 16855452.725202594, 0.0005959183673469389
Stresses 6092202.062585662, 12793059.980133079, 0.0007918367346938775
Stresses 26700840.617351543, 86080364.99669226, 0.0004
Stresses 17891158.170688964, 59544933.8758272, 0.00059

In [8]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [9]:
ac = Aircraft(fixed, [standard_wing] + tail)


In [10]:
for acp in ac.planforms:
    print(acp.mass_cache)

print(ac.fixed.x_cg_min, ac.fixed.x_cg_max, ac.fixed.x_LE_wing)

0.25583403427157053
0.0021845032226573313
0.012122727546318893
1.336 1.403 1.255


# Requirement check for the aircraft

In [11]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [12]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

In [13]:
print("Standard Planform:")
print(f"Tail properties: cr={ac.planforms[1].c_root}, b={ac.planforms[1].span}, sweep={np.rad2deg(ac.planforms[1].sweep_LE_rad)}, ct={ac.planforms[1].c_tip}, x_LE_root={fixed.x_LE_tail}")
print(f"Vertical tail properties: cr={ac.planforms[2].c_root}, b={ac.planforms[2].span}, ct={ac.planforms[2].c_tip}, sweep={np.rad2deg(ac.planforms[2].sweep_LE_rad)}")
print(f"xcg_rang: {fixed.x_cg_min}, {fixed.x_cg_max}")


Standard Planform:
Tail properties: cr=0.11197028240603694, b=0.47587370022565695, sweep=16.86957606886068, ct=0.07837919768422585, x_LE_root=2.4683173177143614
Vertical tail properties: cr=0.24260227854641345, b=0.6786836847529165, ct=0.14556136712784806, sweep=32.97721164076181
xcg_rang: 1.336, 1.403


In [14]:
#TODO: ctrl surface sizing

In [15]:
print(fixed.z_tail_cone - assumptions.main_gear_diameter_wheel /2)


0.07750000000000004


In [ ]:
print(ac.CD0_takeoff, ac.planforms[0].oswald)

0.07928667748719939 0.680337246179267
11.609725538493164
